# Phase 7: Operations Analytics

## Objective

Evaluate Amazon order-status and fulfilment indicators using explicit order-level denominators and descriptive status proxies.

**Scope control:** No delivery duration, SLA, courier-speed ranking, true cancellation rate, true return rate, or warehouse allocation is calculated.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects: print(obj)
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
from src.status_scope import add_status_scope
amazon = pd.read_csv(ROOT / 'data/cleaned/amazon_sale_report_cleaned.csv', low_memory=False)
amazon = add_status_scope(amazon)
amazon['date'] = pd.to_datetime(amazon['date'], errors='coerce')
warehouse = pd.read_csv(ROOT / 'data/cleaned/cloud_warehouse_compersion_chart_cleaned.csv', low_memory=False)
print('Amazon lines:', len(amazon), '| distinct orders:', amazon['order_id'].nunique())


## 1. Grain, order-level status, and mixed-status checks

**Observation:** Amazon rows are line grain, while operational rates use distinct order IDs.

**Evidence:** An order-status table is built by grouping each order and retaining the complete set of line statuses, fulfilment values, channel values, and courier values.

**Interpretation:** The table prevents line duplication from inflating order-level rates and keeps mixed-status orders explicitly unresolved.

**Business implication:** Rates can be reproduced from order-level presence flags, while mixed-status orders become a data-governance finding rather than an assigned lifecycle outcome.

**Limitation:** Status labels are source proxies; no business-approved lifecycle precedence rule exists.

In [ ]:
order_status = amazon.groupby('order_id').agg(status_values=('status', lambda values: tuple(sorted(values.dropna().unique())),), status_count=('status', 'nunique'), fulfilment_values=('fulfilment', lambda values: tuple(sorted(values.dropna().unique())),), channel_values=('sales_channel', lambda values: tuple(sorted(values.dropna().unique())),), courier_values=('courier_status', lambda values: tuple(sorted(values.dropna().unique())),)).reset_index()
order_status['status_label'] = np.where(order_status['status_count'].eq(1), order_status['status_values'].str[0], 'MIXED_STATUS_REQUIRES_RULE')
order_status['order_date'] = order_status['order_id'].map(amazon.groupby('order_id')['date'].min())
order_status['month'] = order_status['order_date'].dt.to_period('M').astype('string')
order_status['has_cancelled_status'] = order_status['status_values'].apply(lambda values: any(str(v).lower() == 'cancelled' for v in values))
order_status['has_return_status'] = order_status['status_values'].apply(lambda values: any('return' in str(v).lower() for v in values))
order_status['has_shipped_status'] = order_status['status_values'].apply(lambda values: any(str(v).lower().startswith('shipped') for v in values))
order_status['has_delivered_status_proxy'] = order_status['status_values'].apply(lambda values: 'Shipped - Delivered to Buyer' in values)
mixed_status_orders = order_status[order_status['status_count'].gt(1)].copy()
mixed_status_detail = amazon[amazon['order_id'].isin(mixed_status_orders['order_id'])].groupby('order_id').agg(status_values=('status', lambda values: tuple(sorted(values.dropna().unique()))), line_count=('order_id', 'size'), sku_values=('sku', lambda values: tuple(sorted(values.dropna().unique())))).reset_index()
print('Orders:', len(order_status), '| mixed-status orders:', len(mixed_status_orders))
display(mixed_status_detail.head(20))
print('Mixed fulfilment orders:', int(amazon.groupby('order_id')['fulfilment'].nunique().gt(1).sum()), '| mixed channel orders:', int(amazon.groupby('order_id')['sales_channel'].nunique().gt(1).sum()))
assert len(order_status) == amazon['order_id'].nunique()
assert len(mixed_status_orders) == int(order_status['status_count'].gt(1).sum())
assert order_status['status_label'].eq('MIXED_STATUS_REQUIRES_RULE').sum() == len(mixed_status_orders)


## 2. Order-status distribution and explicit rate definitions

**Observation:** Status distribution is shown at both line and distinct-order grain.

**Evidence:** Every rate uses the denominator `COUNT(DISTINCT order_id)` across the Amazon reported source: `120,378` orders.

**Definitions:**

- Cancellation status proxy = orders with exact `Cancelled` status / all distinct orders.
- Return status proxy = orders with any status containing `Return` / all distinct orders.
- Shipped status proxy = orders with any status beginning `Shipped` / all distinct orders.
- Delivered status proxy = orders with exact `Shipped - Delivered to Buyer` / all distinct orders.

**Limitation:** These are status proxies, not validated business rates.

In [ ]:
line_status = amazon.groupby('status', dropna=False).agg(line_count=('status', 'size'), distinct_orders=('order_id', 'nunique')).sort_values('line_count', ascending=False)
order_status_distribution = order_status.groupby('status_label', dropna=False).size().to_frame('distinct_orders').sort_values('distinct_orders', ascending=False)
denominator_orders = len(order_status)
rates = pd.Series({'cancellation_status_proxy_rate': order_status['has_cancelled_status'].mean(), 'return_status_proxy_rate': order_status['has_return_status'].mean(), 'shipped_status_proxy_rate': order_status['has_shipped_status'].mean(), 'delivered_status_proxy_rate': order_status['has_delivered_status_proxy'].mean()})
status_time = order_status.groupby('month').agg(distinct_orders=('order_id', 'nunique'), cancelled_orders=('has_cancelled_status', 'sum'), return_orders=('has_return_status', 'sum'), shipped_orders=('has_shipped_status', 'sum'), delivered_orders=('has_delivered_status_proxy', 'sum'))
for flag, numerator in [('cancellation_status_proxy_rate', 'cancelled_orders'), ('return_status_proxy_rate', 'return_orders'), ('shipped_status_proxy_rate', 'shipped_orders'), ('delivered_status_proxy_rate', 'delivered_orders')]: status_time[flag] = status_time[numerator] / status_time['distinct_orders']
display(line_status, order_status_distribution, rates.to_frame('rate'), status_time)
print('Rate denominator: distinct Amazon order IDs =', denominator_orders)
assert line_status['line_count'].sum() == len(amazon)
assert order_status_distribution['distinct_orders'].sum() == denominator_orders
assert status_time['distinct_orders'].sum() == denominator_orders
assert ((rates >= 0) & (rates <= 1)).all()


## 3. Fulfilment method and courier status

**Observation:** Fulfilment and courier labels are available, but timestamps are not.

**Evidence:** The tables show lines and distinct orders by `fulfilment`, `fulfilled_by`, and `courier_status`.

**Interpretation:** These are operational mix indicators, not delivery-speed or SLA measures.

**Business implication:** Use the mix to identify process-review areas and missing-data priorities.

**Limitation:** No order date-to-delivery date duration can be calculated, and courier labels do not establish speed.

In [ ]:
fulfilment_lines = amazon.groupby('fulfilment', dropna=False).agg(line_count=('fulfilment','size'), distinct_orders=('order_id','nunique')).sort_values('distinct_orders', ascending=False)
fulfilled_by_lines = amazon.groupby('fulfilled_by', dropna=False).agg(line_count=('fulfilled_by','size'), distinct_orders=('order_id','nunique')).sort_values('distinct_orders', ascending=False)
courier_lines = amazon.groupby('courier_status', dropna=False).agg(line_count=('courier_status','size'), distinct_orders=('order_id','nunique')).sort_values('distinct_orders', ascending=False)
display(fulfilment_lines, fulfilled_by_lines, courier_lines)
assert fulfilment_lines['line_count'].sum() == len(amazon)
assert courier_lines['line_count'].sum() == len(amazon)


## 4. Platform and category operational differences

**Observation:** Operational proxy rates can be compared by channel and category with sample-size flags.

**Evidence:** Group denominators are distinct orders within each channel/category; `MIN_GROUP_ORDERS = 1,000` is an analytical sufficiency flag, not a business rule. Dimension tables are built at order-plus-dimension grain to avoid many-to-many joins.

**Interpretation:** Differences may reflect product mix, status composition, or sample size; no causal ranking is made.

**Business implication:** Prioritise groups with enough observations for operational review and treat small groups as directional only.

**Limitation:** The Non-Amazon channel has a much smaller sample than Amazon.in, and category populations differ materially.

In [ ]:
MIN_GROUP_ORDERS = 1000
def dimension_order_status(frame, dimension):
    work = frame.copy()
    lowered = work['status'].astype('string').str.lower()
    work['line_cancelled'] = lowered.eq('cancelled')
    work['line_return'] = lowered.str.contains('return', na=False)
    work['line_shipped'] = lowered.str.startswith('shipped', na=False)
    work['line_delivered'] = work['status'].eq('Shipped - Delivered to Buyer')
    return work.groupby(['order_id', dimension], dropna=False).agg(lines=('order_id', 'size'), has_cancelled_status=('line_cancelled', 'any'), has_return_status=('line_return', 'any'), has_shipped_status=('line_shipped', 'any'), has_delivered_status_proxy=('line_delivered', 'any')).reset_index()
def operational_rates(frame, group_col):
    grouped = frame.groupby(group_col, dropna=False)
    result = grouped.agg(distinct_orders=('order_id','nunique'), lines=('order_id','size'), cancelled_orders=('has_cancelled_status','sum'), return_orders=('has_return_status','sum'), shipped_orders=('has_shipped_status','sum'), delivered_orders=('has_delivered_status_proxy','sum'))
    result['cancellation_status_proxy_rate'] = result['cancelled_orders'] / result['distinct_orders']
    result['return_status_proxy_rate'] = result['return_orders'] / result['distinct_orders']
    result['shipped_status_proxy_rate'] = result['shipped_orders'] / result['distinct_orders']
    result['delivered_status_proxy_rate'] = result['delivered_orders'] / result['distinct_orders']
    result['sample_flag'] = np.where(result['distinct_orders'] >= MIN_GROUP_ORDERS, 'sufficient_for_descriptive_comparison', 'small_sample_review_only')
    return result.sort_values('distinct_orders', ascending=False)
channel_order_status = dimension_order_status(amazon, 'sales_channel').rename(columns={'sales_channel': 'channel'})
platform_operations = operational_rates(channel_order_status, 'channel')
category_order_status = dimension_order_status(amazon, 'category')
category_operations = operational_rates(category_order_status, 'category')
display(platform_operations, category_operations)
assert platform_operations['distinct_orders'].sum() == denominator_orders
assert (category_operations['distinct_orders'] <= denominator_orders).all()


## 5. Geography and B2B/B2C status composition

**Observation:** Status-proxy composition can be described by shipping state and B2B flag using source-local order-plus-dimension tables.

**Evidence:** Each group reports distinct orders, line count, proxy counts, proxy rates, and a sample-size flag.

**Interpretation:** Differences may reflect product mix, group size, or status capture; they do not establish an operational cause.

**Business implication:** Use sufficiently observed states and B2B/B2C groups as investigation candidates and collect event-level operational data.

**Limitation:** Small groups are directional only, and no customer, timestamp, or causal operational field is available.

In [ ]:
geo_operations = operational_rates(dimension_order_status(amazon, 'ship_state'), 'ship_state')
b2b_operations = operational_rates(dimension_order_status(amazon, 'b2b'), 'b2b')
display(geo_operations.head(20), b2b_operations)
assert (geo_operations['distinct_orders'] <= denominator_orders).all()
assert (b2b_operations['distinct_orders'] <= denominator_orders).all()


## 6. High-risk review candidates

**Observation:** Categories and SKUs can be flagged for operational review using status-proxy rates and minimum order observations.

**Evidence:** Candidates are descriptive groups with at least 100 distinct orders; no causal risk score or business loss estimate is created.

**Interpretation:** A high proxy rate may reflect data/status process differences or product mix rather than operational failure.

**Business implication:** Review candidate groups with source owners before action.

**Limitation:** SKU-level rates are not stable for very small samples and no financial impact is available.

In [ ]:
order_with_sku = dimension_order_status(amazon, 'sku')
sku_operations = operational_rates(order_with_sku, 'sku')
sku_operations['review_candidate'] = (sku_operations['distinct_orders'] >= 100) & ((sku_operations['cancellation_status_proxy_rate'] > rates['cancellation_status_proxy_rate']) | (sku_operations['return_status_proxy_rate'] > rates['return_status_proxy_rate']))
category_review = category_operations[(category_operations['distinct_orders'] >= 100) & (category_operations['cancellation_status_proxy_rate'] > rates['cancellation_status_proxy_rate'])].copy()
sku_review = sku_operations[sku_operations['review_candidate']].sort_values(['return_status_proxy_rate','cancellation_status_proxy_rate'], ascending=False).head(20)
display(category_review, sku_review)
print('Review candidates are not labelled high-risk operational failures.')


## 7. Warehouse comparison and exclusions

**Observation:** Warehouse provider prices are available as standalone reference rows but cannot be linked to Amazon orders or fulfilment outcomes.

**Evidence:** The warehouse table has no order ID, SKU, date, or common transaction key.

**Interpretation:** A provider-rate comparison may be descriptive, but it cannot explain order-level performance.

**Business implication:** Keep warehouse comparison outside order fulfilment ranking until a transaction-level linkage exists.

**Explicit exclusions:** delivery duration, on-time delivery, SLA compliance, courier speed ranking, true cancellation rate, true return rate, warehouse-attributed performance, and causal platform/category claims.

In [ ]:
display(warehouse)
assert not {'order_id', 'sku', 'date'}.issubset(warehouse.columns)
print('Warehouse data is not joined to operational order records.')


## 8. Reconciliation and limitations

The final checks reconcile line and order status totals, verify order-level deduplication, confirm denominators, examine mixed-status orders, and verify sample-size flags.

In [ ]:
assert int(order_status['has_cancelled_status'].sum()) == int(order_status_distribution.loc['Cancelled', 'distinct_orders'])
assert int(order_status['has_delivered_status_proxy'].sum()) == int(order_status_distribution.loc['Shipped - Delivered to Buyer', 'distinct_orders'])
assert mixed_status_detail['order_id'].nunique() == len(mixed_status_orders)
assert platform_operations['sample_flag'].isin({'sufficient_for_descriptive_comparison','small_sample_review_only'}).all()
assert category_operations['sample_flag'].isin({'sufficient_for_descriptive_comparison','small_sample_review_only'}).all()
print('Operations totals, order deduplication, status denominators, mixed-status checks, and sample flags validated.')
